# Bivariate Spatial Cross-Correlation (Moran's I)

**Aim 2 — Spatial co-localization of cell types across metastasis groups**

For each sample, computes bivariate Moran's I between Cell2Location abundance scores for key cell type pairs. Tests whether spatial co-localization differs by metastasis group.

This provides quantitative evidence for spatial co-localization that motivates the NMF niche analysis.

In [12]:
import scanpy as sc
import numpy as np
import pandas as pd
import os
import warnings
from scipy.spatial import KDTree
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")

# ============================================================
# CONFIGURATION
# ============================================================
SAMPLE_DIR = "/Users/adiallo/Desktop/Thesis/Data_Documents/All_Data/Visium_samples/Visium_Data/samples"
META_PATH = "/Users/adiallo/Desktop/Thesis/Data_Documents/data_all.csv"
OUTPUT_DIR = "/Users/adiallo/Desktop/Thesis/Results/AIM_2_Results/Spatial_CrossCorrelation"
ABUNDANCE_KEY = "means_cell_abundance_w_sf"

os.makedirs(OUTPUT_DIR, exist_ok=True)

## Cell Type Pairs and Mappings

Biologically motivated pairs testing immune infiltration, exclusion, and immunosuppressive niche formation.

In [13]:
# Cell type pairs to test
CELL_TYPE_PAIRS = [
    ("Tumor", "CD8"),
    ("Tumor", "Treg"),
    ("Tumor", "Macrophage"),
    ("Tumor", "CAF"),
    ("CAF", "CD8"),
    ("CAF", "Treg"),
    ("CAF", "Macrophage"),
    ("Tumor", "DC"),
]

# Mapping from broad names to Cell2Location column patterns
BROAD_TO_C2L = {
    "Tumor":      ["Tumor"],
    "CD8":        ["cTNI10", "cTNI11", "cTNI12", "cTNI13", "cTNI14", "cTNI15", "cTNI16"],
    "CD4":        ["cTNI01", "cTNI02", "cTNI03", "cTNI04", "cTNI05", "cTNI06", "cTNI07"],
    "Treg":       ["cTNI08", "cTNI09"],
    "CAF":        ["cS27", "cS28", "cS29", "cS30", "cS31"],
    "Macrophage": ["cM01", "cM02"],
    "DC":         ["cM03", "cM04", "cM05", "cM06", "cM07", "cM08", "cM09"],
    "NK":         ["cTNI23", "cTNI24", "cTNI25"],
    "B_cell":     ["cB1", "cB2", "cB3"],
    "Fibroblast": ["cS21", "cS22", "cS23", "cS24", "cS25"],
}

## Helper Functions

Abundance scoring, spatial weight matrix construction, bivariate Moran's I, and permutation testing.

In [16]:
def get_abundance_score(adata, broad_type):
    """Sum Cell2Location abundances for columns matching a broad cell type."""
    abund = adata.obsm[ABUNDANCE_KEY]
    patterns = BROAD_TO_C2L.get(broad_type, [])
    
    matching_cols = []
    for col in abund.columns:
        for pattern in patterns:
            clean = col.replace("meanscell_abundance_w_sf_", "").replace("means_cell_abundance_w_sf_", "")
            if clean.startswith(pattern) or pattern in clean:
                matching_cols.append(col)
                break
    
    if len(matching_cols) == 0:
        return None
    
    return abund[matching_cols].sum(axis=1).values


def spatial_weights_knn(coords, k=6):
    """Build row-normalized spatial weight matrix using k nearest neighbors."""
    tree = KDTree(coords)
    dists, indices = tree.query(coords, k=k+1)  # +1 for self
    
    n = len(coords)
    W = np.zeros((n, n))
    for i in range(n):
        neighbors = indices[i, 1:]  # skip self
        W[i, neighbors] = 1.0
    
    # Row-normalize
    row_sums = W.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1
    W = W / row_sums
    
    return W


def bivariate_morans_i(x, y, W):
    """
    Compute bivariate Moran's I between variables x and y.
    Measures correlation between x and the spatial lag of y.
    """
    n = len(x)
    
    # Standardize
    zx = (x - x.mean()) / (x.std() + 1e-10)
    zy = (y - y.mean()) / (y.std() + 1e-10)
    
    # Spatial lag of y
    Wy = W @ zy
    
    # Bivariate Moran's I = correlation of x with spatial lag of y
    I = np.sum(zx * Wy) / n
    
    return I


def permutation_test(x, y, W, n_perms=9999):
    """Permutation test for bivariate Moran's I."""
    observed = bivariate_morans_i(x, y, W)
    
    count = 0
    for _ in range(n_perms):
        y_perm = np.random.permutation(y)
        I_perm = bivariate_morans_i(x, y_perm, W)
        if abs(I_perm) >= abs(observed):
            count += 1
    
    p_value = (count + 1) / (n_perms + 1)
    return observed, p_value

## Load Clinical Metadata

In [17]:
print("=" * 70)
print("BIVARIATE SPATIAL CROSS-CORRELATION ANALYSIS")
print("=" * 70)

clinical = pd.read_csv(META_PATH)
clinical.columns = clinical.columns.str.strip()
clinical["deident"] = clinical["deident"].astype(str)

def get_mets_group(row):
    if row.get("Distant_Mets") in [True, "TRUE", "True", 1]:
        return "Distant_Mets"
    elif row.get("ln_only") in [True, "TRUE", "True", 1]:
        return "LN_Mets"
    else:
        return "No_Mets"

clinical["mets_group"] = clinical.apply(get_mets_group, axis=1)
print(f"Mets group distribution:")
print(clinical["mets_group"].value_counts())

BIVARIATE SPATIAL CROSS-CORRELATION ANALYSIS
Mets group distribution:
mets_group
No_Mets         38
Distant_Mets    19
Name: count, dtype: int64


## Compute Bivariate Moran's I Per Sample

For each sample and cell type pair, compute bivariate Moran's I with permutation test (999 permutations).

**Note:** This may take 30-60 minutes for all 41 samples.

In [18]:
print("Computing bivariate Moran's I for each sample and cell type pair...")
print("-" * 70)

results = []

h5ad_files = [f for f in h5ad_files if f.replace('.h5ad', '').isdigit()]

for h5ad_file in h5ad_files:
    sid = h5ad_file.replace(".h5ad", "")
    adata = sc.read_h5ad(os.path.join(SAMPLE_DIR, h5ad_file))
    
    coords = adata.obsm["spatial"]
    
    # Build spatial weight matrix (6 nearest neighbors for Visium hexagonal grid)
    W = spatial_weights_knn(coords, k=6)
    
    print(f"Sample {sid}: {adata.n_obs} spots")
    
    for type_a, type_b in CELL_TYPE_PAIRS:
        score_a = get_abundance_score(adata, type_a)
        score_b = get_abundance_score(adata, type_b)
        
        if score_a is None or score_b is None:
            print(f"    {type_a}-{type_b}: missing columns, skipped")
            continue
        
        # Skip if no variance
        if score_a.std() < 1e-10 or score_b.std() < 1e-10:
            print(f"    {type_a}-{type_b}: no variance, skipped")
            continue
        
        I_val, p_val = permutation_test(score_a, score_b, W, n_perms=999)
        
        # Get mets group for this sample
        mets_row = clinical[clinical["deident"] == sid]
        mets_group = mets_row["mets_group"].values[0] if len(mets_row) > 0 else "Unknown"
        
        results.append({
            "sample_id": sid,
            "type_a": type_a,
            "type_b": type_b,
            "pair": f"{type_a}–{type_b}",
            "morans_I": I_val,
            "p_value": p_val,
            "significant": p_val < 0.05,
            "mets_group": mets_group,
            "n_spots": adata.n_obs,
        })
        
        sig_str = "*" if p_val < 0.05 else " "
        print(f"    {type_a}-{type_b}: I={I_val:.4f}, p={p_val:.3f} {sig_str}")

results_df = pd.DataFrame(results)
results_df.to_csv(os.path.join(OUTPUT_DIR, "bivariate_morans_i_results.csv"), index=False)
print(f"Results saved: {len(results_df)} measurements")

Computing bivariate Moran's I for each sample and cell type pair...
----------------------------------------------------------------------
Sample 0: 6658 spots
    Tumor-CD8: I=0.5987, p=0.001 *
    Tumor-Treg: I=0.6532, p=0.001 *
    Tumor-Macrophage: I=0.2329, p=0.001 *
    Tumor-CAF: I=0.2274, p=0.001 *
    CAF-CD8: I=0.2630, p=0.001 *
    CAF-Treg: I=0.2726, p=0.001 *
    CAF-Macrophage: I=0.5442, p=0.001 *
    Tumor-DC: I=0.6277, p=0.001 *
Sample 10: 6138 spots
    Tumor-CD8: I=0.6794, p=0.001 *
    Tumor-Treg: I=0.7131, p=0.001 *
    Tumor-Macrophage: I=0.2271, p=0.001 *
    Tumor-CAF: I=-0.0879, p=0.001 *
    CAF-CD8: I=0.0753, p=0.001 *
    CAF-Treg: I=0.0495, p=0.001 *
    CAF-Macrophage: I=0.1690, p=0.001 *
    Tumor-DC: I=0.6138, p=0.001 *
Sample 100: 7194 spots
    Tumor-CD8: I=0.2070, p=0.001 *
    Tumor-Treg: I=0.2741, p=0.001 *
    Tumor-Macrophage: I=-0.1555, p=0.001 *
    Tumor-CAF: I=-0.0049, p=0.640  
    CAF-CD8: I=0.1510, p=0.001 *
    CAF-Treg: I=0.1281, p=0.001 *

## Summary Statistics

In [19]:
print(f"{'=' * 70}")
print("SUMMARY")
print(f"{'=' * 70}")

for pair in results_df["pair"].unique():
    sub = results_df[results_df["pair"] == pair]
    n_sig = sub["significant"].sum()
    mean_I = sub["morans_I"].mean()
    print(f"{pair}:")
    print(f"    Mean Moran's I = {mean_I:.4f}")
    print(f"    Significant in {n_sig}/{len(sub)} samples")
    
    for mg in ["No_Mets", "LN_Mets", "Distant_Mets"]:
        mg_sub = sub[sub["mets_group"] == mg]
        if len(mg_sub) > 0:
            print(f"    {mg}: mean I = {mg_sub['morans_I'].mean():.4f} (n={len(mg_sub)})")

SUMMARY
Tumor–CD8:
    Mean Moran's I = 0.4122
    Significant in 39/41 samples
    No_Mets: mean I = 0.3912 (n=31)
    Distant_Mets: mean I = 0.4772 (n=10)
Tumor–Treg:
    Mean Moran's I = 0.4704
    Significant in 41/41 samples
    No_Mets: mean I = 0.4532 (n=31)
    Distant_Mets: mean I = 0.5237 (n=10)
Tumor–Macrophage:
    Mean Moran's I = 0.0839
    Significant in 39/41 samples
    No_Mets: mean I = 0.0683 (n=31)
    Distant_Mets: mean I = 0.1324 (n=10)
Tumor–CAF:
    Mean Moran's I = -0.0605
    Significant in 36/41 samples
    No_Mets: mean I = -0.0511 (n=31)
    Distant_Mets: mean I = -0.0900 (n=10)
CAF–CD8:
    Mean Moran's I = 0.1284
    Significant in 37/41 samples
    No_Mets: mean I = 0.1456 (n=31)
    Distant_Mets: mean I = 0.0749 (n=10)
CAF–Treg:
    Mean Moran's I = 0.1084
    Significant in 39/41 samples
    No_Mets: mean I = 0.1262 (n=31)
    Distant_Mets: mean I = 0.0532 (n=10)
CAF–Macrophage:
    Mean Moran's I = 0.3162
    Significant in 40/41 samples
    No_Mets: 

## Statistical Tests: Mets Group Differences

Kruskal-Wallis across all three groups, plus pairwise Mann-Whitney for Distant vs No and LN vs No.

In [23]:
print(f"{'=' * 70}")
print("METS GROUP COMPARISONS (Kruskal-Wallis)")
print(f"{'=' * 70}")

stat_results = []

for pair in results_df["pair"].unique():
    sub = results_df[results_df["pair"] == pair]
    
    groups = {}
    for mg in ["No_Mets", "LN_Mets", "Distant_Mets"]:
        vals = sub[sub["mets_group"] == mg]["morans_I"].values
        if len(vals) >= 2:
            groups[mg] = vals
    
    if len(groups) >= 2:
        group_vals = list(groups.values())
        kw_stat, kw_p = stats.kruskal(*group_vals)
        print(f"{pair}: H={kw_stat:.3f}, p={kw_p:.4f}")
        
        stat_results.append({
            "pair": pair,
            "test": "Kruskal-Wallis",
            "statistic": kw_stat,
            "p_value": kw_p,
        })
        
        # Pairwise: Distant vs No Mets
        if "Distant_Mets" in groups and "No_Mets" in groups:
            u_stat, u_p = stats.mannwhitneyu(groups["Distant_Mets"], groups["No_Mets"], alternative="two-sided")
            print(f"    Distant vs No: U={u_stat:.1f}, p={u_p:.4f}")
            stat_results.append({"pair": pair, "test": "Distant_vs_No", "statistic": u_stat, "p_value": u_p})
        
        # Pairwise: LN vs No Mets
        if "LN_Mets" in groups and "No_Mets" in groups:
            u_stat, u_p = stats.mannwhitneyu(groups["LN_Mets"], groups["No_Mets"], alternative="two-sided")
            print(f"    LN vs No: U={u_stat:.1f}, p={u_p:.4f}")
            stat_results.append({"pair": pair, "test": "LN_vs_No", "statistic": u_stat, "p_value": u_p})

pd.DataFrame(stat_results).to_csv(os.path.join(OUTPUT_DIR, "mets_group_statistics.csv"), index=False)
print(f"Statistics saved.")


METS GROUP COMPARISONS (Kruskal-Wallis)
Tumor–CD8: H=0.944, p=0.3313
    Distant vs No: U=187.0, p=0.3389
Tumor–Treg: H=1.065, p=0.3020
    Distant vs No: U=189.0, p=0.3091
Tumor–Macrophage: H=0.299, p=0.5848
    Distant vs No: U=173.0, p=0.5952
Tumor–CAF: H=0.075, p=0.7847
    Distant vs No: U=146.0, p=0.7964
CAF–CD8: H=0.623, p=0.4299
    Distant vs No: U=129.0, p=0.4388
CAF–Treg: H=0.488, p=0.4850
    Distant vs No: U=132.0, p=0.4946
CAF–Macrophage: H=5.184, p=0.0228
    Distant vs No: U=80.0, p=0.0237
Tumor–DC: H=1.626, p=0.2023
    Distant vs No: U=197.0, p=0.2077
Statistics saved.


## Boxplots: Moran's I by Metastasis Group

In [24]:
METS_ORDER = ["No_Mets", "LN_Mets", "Distant_Mets"]
METS_PALETTE = {"No_Mets": "#00AFBB", "LN_Mets": "#E7B800", "Distant_Mets": "#FC4E07"}

pairs_to_plot = results_df["pair"].unique()
n_pairs = len(pairs_to_plot)
n_cols = 4
n_rows = int(np.ceil(n_pairs / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 5 * n_rows), facecolor="white")
axes = axes.flatten() if n_pairs > 1 else [axes]

for idx, pair in enumerate(pairs_to_plot):
    ax = axes[idx]
    sub = results_df[results_df["pair"] == pair]
    
    sns.boxplot(data=sub, x="mets_group", y="morans_I", order=METS_ORDER,
                palette=METS_PALETTE, ax=ax, width=0.6)
    sns.stripplot(data=sub, x="mets_group", y="morans_I", order=METS_ORDER,
                  color="black", alpha=0.5, size=4, ax=ax)
    
    ax.set_title(pair, fontsize=12, fontweight="bold")
    ax.set_xlabel("")
    ax.set_ylabel("Bivariate Moran's I")
    ax.axhline(y=0, color="grey", linestyle="--", alpha=0.5)
    ax.tick_params(axis="x", rotation=30)

# Hide unused axes
for idx in range(len(pairs_to_plot), len(axes)):
    axes[idx].set_visible(False)

plt.suptitle("Spatial Cross-Correlation by Metastasis Group
(Bivariate Moran's I — Cell2Location abundances)",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "morans_i_boxplots_by_mets.png"), dpi=200, bbox_inches="tight")
plt.show()
print(f"Plot saved to {OUTPUT_DIR}")

SyntaxError: unterminated string literal (detected at line 31) (2033868454.py, line 31)

## Heatmap: Mean Moran's I by Pair and Mets Group

In [ ]:
pivot = results_df.pivot_table(index="pair", columns="mets_group", values="morans_I", aggfunc="mean")
pivot = pivot.reindex(columns=METS_ORDER)

fig, ax = plt.subplots(figsize=(8, 6), facecolor="white")
sns.heatmap(pivot, annot=True, fmt=".3f", cmap="RdBu_r", center=0, ax=ax,
            linewidths=0.5, linecolor="white")
ax.set_title("Mean Bivariate Moran's I
(Cell type spatial co-localization by mets group)",
             fontsize=13, fontweight="bold")
ax.set_ylabel("")
ax.set_xlabel("")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "morans_i_heatmap.png"), dpi=200, bbox_inches="tight")
plt.show()

print(f"
All results saved to {OUTPUT_DIR}")
print("Done!")

In [25]:
visium_ids = [str(s) for s in [0,3,6,10,11,13,14,18,28,29,30,34,36,38,40,46,49,51,52,60,
                                72,73,74,80,81,83,86,91,96,97,100,102,106,107,111,116,117,
                                119,122,127,128]]

visium_clinical = clinical[clinical["deident"].isin(visium_ids)].copy()
print("Visium subset mets distribution:")
print(visium_clinical["mets_group"].value_counts())
print()
print("Distant_Mets values:")
print(visium_clinical["Distant_Mets"].value_counts(dropna=False))
print()
print("ln_only values:")
print(visium_clinical["ln_only"].value_counts(dropna=False))
print()
print("any_mets values:")
print(visium_clinical["any_mets"].value_counts(dropna=False))

Visium subset mets distribution:
mets_group
No_Mets         31
Distant_Mets    10
Name: count, dtype: int64

Distant_Mets values:
Distant_Mets
False    31
True     10
Name: count, dtype: int64

ln_only values:
ln_only
F    29
T    12
Name: count, dtype: int64

any_mets values:
any_mets
T    22
F    19
Name: count, dtype: int64
